# 03 — Clinical Results

Reproduce Tables 3–8 and Figures 2–7 from the paper using saved result JSON files.
Run `scripts/evaluate.py` first to generate the JSON outputs.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

RESULTS = Path('../results')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

## Table 3 — LOSO Cross-Validation (Neonatal Pilot)

In [ ]:
for k in [1, 10]:
    path = RESULTS / f'loso_{k}shot.json'
    if not path.exists():
        print(f'{path.name} not found — run evaluate.py first.')
        continue
    data = json.loads(path.read_text())
    print(f"\n{k}-shot LOSO Results:")
    print(f"  Mean AUC  : {data['mean_auc']:.3f} ± {data['std_auc']:.3f}")
    print(f"  95% CI    : [{data['ci'][0]:.3f}, {data['ci'][1]:.3f}]")
    print(f"  Min / Max : {data['min_auc']:.3f} / {data['max_auc']:.3f}")
    pm = data['pooled_metrics']
    print(f"  Accuracy  : {pm['accuracy']*100:.1f}%")
    print(f"  F1 macro  : {pm['f1_macro']:.3f}")
    print(f"  ECE       : {pm.get('ece', float('nan')):.3f}")

## Figure 2 — Per-Subject LOSO AUC Distribution

In [ ]:
for k in [1, 10]:
    path = RESULTS / f'loso_{k}shot.json'
    if not path.exists():
        continue
    data = json.loads(path.read_text())
    per_s = data['per_subject']
    aucs  = sorted([s['auc'] for s in per_s])

    fig, ax = plt.subplots(figsize=(10, 4))
    bar_colors = ['#E53935' if a < 0.7 else '#FFA726' if a < 0.8 else '#43A047'
                  for a in aucs]
    ax.bar(range(len(aucs)), aucs, color=bar_colors, width=0.7)
    ax.axhline(data['mean_auc'], color='navy', ls='--', lw=1.5,
               label=f"Mean = {data['mean_auc']:.3f}")
    ax.axhline(0.8, color='gray', ls=':', lw=1.2, label='AUC = 0.8 threshold')
    ax.set_xlabel('Subject (sorted by AUC)')
    ax.set_ylabel('LOSO AUC')
    ax.set_title(f'Per-Subject LOSO AUC — {k}-shot (Neonatal Pilot, n={len(aucs)})')
    ax.set_ylim(0.4, 1.0)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'loso_per_subject_{k}shot.png', dpi=150)
    plt.show()

## Table 2 — MMD Domain Gap

In [ ]:
mmd_path = RESULTS / 'mmd_domain_gap.json'
if mmd_path.exists():
    mmd = json.loads(mmd_path.read_text())
    print('MMD Domain Gap (Adult → Neonatal):')
    print(f"{'Modality':<12} {'Pre-adaptation':>16} {'Post-adaptation':>16}")
    print('-' * 46)
    pre  = mmd.get('pre_adaptation', {})
    post = mmd.get('post_adaptation', {})
    for m in ['video', 'audio', 'physio', 'fused']:
        pre_v  = pre.get(m, float('nan'))
        post_v = post.get(m, float('nan'))
        print(f"{m.capitalize():<12} {pre_v:>16.4f} {post_v:>16.4f}")
else:
    print('mmd_domain_gap.json not found. Run scripts/compute_mmd.py first.')

## Reliability Diagram (Calibration)

In [ ]:
# Simulated calibration curve for illustration
# Replace with actual y_prob / y_true arrays from your evaluation run

bins = np.linspace(0.05, 0.95, 10)
# Ideal calibration
ideal = bins
# Our model (paper ECE = 0.06)
ours  = bins + np.random.normal(0, 0.03, len(bins))
ours  = np.clip(ours, 0, 1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Perfect calibration')
ax.plot(bins, ours, 'o-', color='#1E88E5', lw=2, ms=6, label='Our model (ECE=0.06)')
ax.fill_between(bins, ideal - 0.05, ideal + 0.05, alpha=0.15, color='gray')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Reliability Diagram (Adult Domain)')
ax.legend()
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('reliability_diagram.png', dpi=150)
plt.show()